# Eksplorasi: Pembuatan Vector Store (ChromaDB) dengan SumoPod API

Notebook ini memuat kredensial dari `.env`, memotong teks dari PDF, menghasilkan *embeddings* (menggunakan API yang kompatibel dengan OpenAI dari SumoPod), dan menyimpannya ke *database* Chroma lokal.

In [1]:
import os
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma

# Memuat variabel lingkungan dari file .env (yang berada di satu tingkat ke atas dari folder notebooks)
load_dotenv('../.env')

# Mengonfigurasi modul OpenAIEmbeddings agar mengarah ke SumoPod API
embeddings = OpenAIEmbeddings(
    openai_api_key=os.getenv("SUMOPOD_API_KEY"),
    openai_api_base=os.getenv("SUMOPOD_API_BASE"),
    model="text-embedding-3-small" # Model embedding yang disediakan/didukung oleh SumoPod
)
print("Embedding model berhasil dikonfigurasi.")

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_26964\1540794039.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyMuPDFLoader


Embedding model berhasil dikonfigurasi.


### 1. Memuat dan Memotong Teks (Seperti sebelumnya)

In [2]:
# Path PDF (Pastikan file ada, dan gunakan raw string 'r' untuk Windows)
pdf_path = r"D:\Self Project\SumoPod API\Profile\projects\ai_document_assistant\data\raw\ADPU441003 - Kebijakan Publik.pdf"

loader = PyMuPDFLoader(pdf_path)

# Memuat seluruh halaman tanpa dibatasi
all_pages = loader.load()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)
chunks = text_splitter.split_documents(all_pages)
print(f"Menghasilkan {len(chunks)} chunks teks untuk di-embed.")


Menghasilkan 957 chunks teks untuk di-embed.


### 2. Menyimpan Vektor ke ChromaDB
Proses ini akan mengirimkan teks ke SumoPod API untuk diubah menjadi vektor (melalui internet), lalu menyimpannya ke folder lokal `vector_store/` agar kelak tidak perlu di-*embed* ulang.

In [3]:
# Menentukan direktori penyimpanan lokal
persist_directory = "../vector_store"

# Memasukkan chunks ke ChromaDB dan menyimpannya ke disk
# Catatan: Proses ini mungkin memakan waktu beberapa detik tergantung kecepatan koneksi ke API
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=persist_directory
)
print(f"Vektor berhasil disimpan di {persist_directory}")

Vektor berhasil disimpan di ../vector_store


### 3. Menguji Pencarian (Similarity Search)
Kita menguji apakah kita dapat menemukan bagian teks yang relevan dari *database* vektor berdasarkan pertanyaan dalam bahasa natural.

In [5]:
query = "Apa definisi atau pengertian dari kebijakan publik menurut dokumen ini?"

# Mencari 3 chunk yang paling relevan (k=3)
docs = vectorstore.similarity_search(query, k=3)

print(f"Hasil pencarian untuk: '{query}'\n")
for i, doc in enumerate(docs):
    # Metadata 'page' menyimpan informasi dari halaman berapa teks ini berasal
    print(f"--- Hasil {i+1} (Berasal dari Halaman {doc.metadata.get('page', 'Unknown')}) ---")
    print(doc.page_content[:300] + "...\n")

Hasil pencarian untuk: 'Apa definisi atau pengertian dari kebijakan publik menurut dokumen ini?'

--- Hasil 1 (Berasal dari Halaman 10) ---
Secara singkat 
atau sederhana kebijakan publik 
itu dapat diartikan sebagai 
‘tindakan yang dilakukan oleh pemerintah’ atau ‘aktivitas-aktivitas yang dilakukan 
pemerintah’ (‘the actions of government’). Tentunya pengertian seperti ini terlampau 
ringkas untuk dapat menjelaskan substansi atau isi k...

--- Hasil 2 (Berasal dari Halaman 18) ---
Secara sederhana kebijakan publik dapat diartikan 
sebagai 
“apa 
saja yang 
dilakukan oleh pemerintah” (the actions of government). Sebagai orang yang baru 
mempelajari kebijakan publik, tentunya Anda ingin mengetahui dan memahami artinya 
lebih dalam. Setiap tulisan tentang kebijakan publik tentu ...

--- Hasil 3 (Berasal dari Halaman 8) ---
| ini, Anda diharapkan dapat menjelaskan: 
1) 
arti dan makna beberapa definisi kebijakan publik; 
2) 
 adanya nuansa dan hubungan antara kebijakan publik dan kepentin

In [6]:
# Mengambil seluruh data dari database vektor lokal
all_data = vectorstore.get()

# all_data['metadatas'] berisi kumpulan metadata dari tiap chunk teks
# Kita ambil nomor 'page' dan menyimpannya di dalam "set" agar tidak ada duplikat
halaman_unik = set()

for metadata in all_data['metadatas']:
    if metadata and 'page' in metadata:
        halaman_unik.add(metadata['page'])

# Mengurutkan nomor halaman dari yang terkecil ke terbesar
halaman_terurut = sorted(list(halaman_unik))

print(f"Total chunk (potongan teks) di database: {len(all_data['ids'])}")
print(f"Total halaman yang di-vektorisasi: {len(halaman_unik)} halaman")
if len(halaman_terurut) > 0:
    print(f"Halaman yang tercakup: {halaman_terurut[0]} sampai {halaman_terurut[-1]}")


Total chunk (potongan teks) di database: 975
Total halaman yang di-vektorisasi: 315 halaman
Halaman yang tercakup: 0 sampai 320
